In [2]:
import os
import openai
import dotenv
import logging

from utilities import Utilities

dotenv.load_dotenv()

True

In [3]:
use_azure_active_directory = False  # Set this flag to True if you are using Azure Active Directory

if not use_azure_active_directory:
    endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
    api_key = os.environ["AZURE_OPENAI_API_KEY"]
    # set the deployment model we want to use
    deploymentid = "gpt-4o"
    # apiversion = "2023-09-01-preview"
    apiversion = "2024-08-01-preview"

    print(endpoint)    
    print(api_key)

# Generic AI client to extract information from user prompt
    client1 = openai.AzureOpenAI(
        azure_endpoint=endpoint,
        api_key=api_key,
        api_version=apiversion        
    )

    print(client1)
# # AI extension client to work with custom data
    client2 = openai.AzureOpenAI(
        base_url=f"{endpoint}/openai/deployments/{deploymentid}/extensions",
        api_key=api_key,
        api_version="2023-12-01-preview"
    )

https://rayanazureaihu6537175092.openai.azure.com/
de399a5dda5a4a5e90983e3c135a32bc


Authentication using Azure Active Directory
ToDo: Need to assign RBAC as `Cognitive Services OpenAI User or Contributor` role instead of having to use key-based auth.

In [4]:
import json

# Method to compose the ingredients based on user's choice and/or picked based on meijer store criteria 
def get_items_for_recipe(storeid, user_ingredients) -> str:
    """Get the ingredients as comma separated string using Meijer products and user suggested items if specified"""
    try:                
        # Todo: logic to read store specific produce items under promo / excess / frequently sold
        pdt_file="meijer_products.csv"
        mjr_pdts = Utilities().get_meijer_products(pdt_file, storeid)
        
        if user_ingredients and mjr_pdts:            
            return ', '.join([user_ingredients, mjr_pdts])
        elif user_ingredients:
            return user_ingredients
        else:
            return mjr_pdts
    except Exception as e:
        logging.error(f"Exception in get_recipe : {e}")
        return e    


Tools Specification

In [22]:
tools=[
    {
        "type": "function",
        "function":
        {
            "name": "get_items_for_recipe",
            "description": "Get the items for recipe based on user input and meijer store product",
            "parameters": {
                "type":"object",
                "properties": {
                    "storeid": {
                        "type": "integer",
                        "description": "The store id of Meijer store if specified, eg: 21 for Alpine",
                    },
                    "user_ingredients": {
                        "type": "string",
                        "description": f"""
                                Comma separated list of ingredients for the recipe extracted from user's input.
                                """
                    }
                },
                "required": [],
            }
        }
    }
]

extract_assistant_message = { "role": "assistant", "content": "You are an AI assistant. Please extract store info and names of items from user prompt. Please do not assume storeid or ingredients if not specified explicitly." }
# Below user prompt can signify 
# 1. a user adding items to cart where the recipe will be based on their cart items / user uses smart devices for managing their shopping list
# 2. meijer specific products chosen based on criteria at the store like items in promotion / high inventory items at the (can be store specific).
user_prompt_message = { "role": "user", "content": "Tell me a dad joke" }
#user_prompt_message = { "role": "user", "content": "I am at a Meijer store 21 and checking to see what I can make with avocado and pasta" }
messages=[extract_assistant_message, user_prompt_message]
print (messages)

api_msg_input = { "messages": messages }
print(api_msg_input)


[{'role': 'assistant', 'content': 'You are an AI assistant. Please extract store info and names of items from user prompt. Please do not assume storeid or ingredients if not specified explicitly.'}, {'role': 'user', 'content': 'Tell me a dad joke'}]
{'messages': [{'role': 'assistant', 'content': 'You are an AI assistant. Please extract store info and names of items from user prompt. Please do not assume storeid or ingredients if not specified explicitly.'}, {'role': 'user', 'content': 'Tell me a dad joke'}]}


In [25]:
# 2023-09-01-preview

item_response = client1.chat.completions.create(
    model=deploymentid,
    messages=messages,    
    tools=tools,
    tool_choice= "auto" #{"type": "function", "function": { "name": "get_items_for_recipe"} }
)

item_message = item_response.choices[0].message
print(item_message)
# print(item_message.tool_calls[0].function.arguments)


ChatCompletionMessage(content="Why don't skeletons fight each other?\n\nThey don't have the guts.", role='assistant', function_call=None, tool_calls=None)


In [25]:
# 2024-02-15-preview
import requests

#configuration
chaturl=f"{endpoint}openai/deployments/{deploymentid}/chat/completions?api-version={apiversion}"
headers = {
    "Content-Type": "application/json",
    "api-key": api_key
}

# send request
try:
    response = requests.post(chaturl, headers=headers, json=api_msg_input)
    response.raise_for_status()   # Will raise an HTTPError if the HTTP request returned an unsuccessful status code
except requests.RequestException as e:
    raise SystemExit(f"Failed to make request. Error: {e}")

print(json.loads(response.content)['choices'][0]['message']['content'])


**Store Information:**
- Store Name: Meijer
- Store Number: 21

**Items:**
- Avocado
- Pasta


In [5]:
recipe_ingredients =None

# check if the model wanted to call a function
if dict(item_message).get('tool_calls'):
    available_functions = {
        "get_items_for_recipe": get_items_for_recipe
    }

    # extracting the functions
    for tool_call in item_message.tool_calls:
        function_name = tool_call.function.name
        function_to_call = available_functions[function_name]
        function_args = json.loads(tool_call.function.arguments)

        function_resp = function_to_call(
            storeid=function_args.get('storeid'),
            user_ingredients=function_args.get('user_ingredients')
        )
        print(f"function_args : {function_args} - storeid :{function_args.get('storeid')}  - user_ingredients: {function_args.get('user_ingredients')}")
        recipe_ingredients=function_resp    
else:
    recipe_ingredients=get_items_for_recipe()

print(f"recipe_ingredients : {recipe_ingredients}")

function_args : {'storeid': 21} - storeid :21  - user_ingredients: None
recipe_ingredients : milk, eggs, chicken, shrimp, fish


In [6]:
messages.clear()

messages.extend(
    [
        { "role": "system", "content": "You are a Meijer AI bot that can suggest a recipe based on the time of the day and ingredients specified by user."},    
        { "role": "user", "content": f"Please suggests one recipe that I can make using some of the following ingredients: {recipe_ingredients}"}
    ]
)

print(messages)
recipe_response = client2.chat.completions.create(
    model=deploymentid,
    messages=messages,
    extra_body={
        "dataSources": [
            {
                "type": "AzureCognitiveSearch",
                "parameters": {
                    "endpoint": os.environ["SEARCH_ENDPOINT"],
                    "key": os.environ["SEARCH_API_KEY"],
                    "indexName": os.environ["SEARCH_INDEX_NAME"]
                }
            }
        ]
    }
)

recipe_message = recipe_response.choices[0].message
print(recipe_message)
print(recipe_message.content)

[{'role': 'system', 'content': 'You are a Meijer AI bot that can suggest a recipe based on the time of the day and ingredients specified by user.'}, {'role': 'user', 'content': 'Please suggests one recipe that I can make using some of the following ingredients: milk, eggs, chicken, shrimp, fish'}]
ChatCompletionMessage(content="Here's a recipe suggestion that you can make using some of the ingredients you mentioned:\n\nYogurt & White Bean Chicken Chili [doc3]\n- Ingredients:\n  - 1 cup Chobani® Whole Milk Plain Greek Yogurt\n  - 3 Tbsp Canola oil\n  - ¾ cup Chopped yellow onion\n  - 1 tsp Minced garlic\n  - 1 lb Boneless, skinless chicken breast diced to ½-inch cubes\n  - 1 tsp Ground cumin\n  - 1 tsp Dried oregano\n  - ¼ tsp Cayenne pepper\n  - ½ tsp Kosher salt\n  - ½ tsp Fresh ground black pepper\n  - 3 cups Canned and rinsed cannellini beans\n  - ¼ cup Canned chopped green chili peppers\n  - 2 cups Low-sodium chicken stock\n\n- Preparation:\n  1. Heat oil in a large pot over medium